# **Install dependency**

In [0]:
%pip install --quiet "psycopg[binary]"

## **PostgreSQL initial snapshot extractor**

In [0]:

# ============================================================
# BANGLAPAY | PHASE 1
# POSTGRESQL BRONZE INITIAL SNAPSHOT
#
# Environment : DEV
# Source      : Neon PostgreSQL
# Tables      : 22
# Data volume : Deterministic 20%
# Landing     : Unity Catalog Managed Volume
# Format      : NDJSON
#
# Source access: READ ONLY
# Snapshot type: INITIAL_SNAPSHOT
# ============================================================

import os
import json
import uuid
import base64

from datetime import date, datetime, timezone
from decimal import Decimal
from uuid import UUID

import psycopg

from psycopg import sql
from psycopg.rows import dict_row

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

ENVIRONMENT = "dev"

SAMPLE_PERCENT = 20
EXPECTED_TABLE_COUNT = 22

DATABASE_NAME = "banglapay"

SECRET_SCOPE = "banglapay-ingestion"

VOLUME_ROOT = (
    "/Volumes/banglapay/ops/batch_landing/postgres"
)

MANIFEST_TABLE = (
    "banglapay.ops.postgres_batch_manifest"
)

FETCH_SIZE = 1000

SNAPSHOT_TYPE = "INITIAL_SNAPSHOT"

BATCH_ID = str(uuid.uuid4())

EXTRACTED_AT = datetime.now(timezone.utc)


if ENVIRONMENT == "dev":
    assert SAMPLE_PERCENT == 20
else:
    raise ValueError(
        "This notebook is currently restricted to DEV."
    )


# ============================================================
# 2. LOAD POSTGRESQL SECRETS
# ============================================================

JDBC_URL = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_JDBC_URL"
).strip()

PG_USERNAME = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_USERNAME"
).strip()

PG_PASSWORD = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_PASSWORD"
).strip()


if JDBC_URL.startswith("jdbc:"):
    PG_URL = JDBC_URL[len("jdbc:"):]

elif JDBC_URL.startswith(
    ("postgresql://", "postgres://")
):
    PG_URL = JDBC_URL

else:
    raise ValueError(
        "NEON_JDBC_URL has an unsupported format."
    )


if not all([PG_URL, PG_USERNAME, PG_PASSWORD]):
    raise RuntimeError(
        "PostgreSQL connection configuration is incomplete."
    )


# ============================================================
# 3. JSON SERIALIZATION
#
# Preserve exact values without converting source fields
# into business-level representations.
# ============================================================

def json_safe(value):

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, Decimal):
        return str(value)

    if isinstance(value, UUID):
        return str(value)

    if isinstance(value, memoryview):
        value = value.tobytes()

    if isinstance(value, bytes):
        return {
            "__encoding": "base64",
            "value": base64.b64encode(value).decode("ascii")
        }

    raise TypeError(
        f"Unsupported PostgreSQL value type: "
        f"{type(value).__name__}"
    )


# ============================================================
# 4. DISCOVER ALL TABLES AND PRIMARY KEYS
# ============================================================

TABLE_INVENTORY_QUERY = """
SELECT
    t.table_schema,
    t.table_name,

    array_agg(
        kcu.column_name::text
        ORDER BY kcu.ordinal_position
    ) FILTER (
        WHERE kcu.column_name IS NOT NULL
    ) AS primary_key_columns

FROM information_schema.tables t

LEFT JOIN information_schema.table_constraints tc

    ON tc.table_schema = t.table_schema
   AND tc.table_name = t.table_name
   AND tc.constraint_type = 'PRIMARY KEY'

LEFT JOIN information_schema.key_column_usage kcu

    ON kcu.constraint_catalog = tc.constraint_catalog
   AND kcu.constraint_schema = tc.constraint_schema
   AND kcu.constraint_name = tc.constraint_name
   AND kcu.table_schema = tc.table_schema
   AND kcu.table_name = tc.table_name

WHERE t.table_type = 'BASE TABLE'

  AND t.table_schema NOT IN (
      'pg_catalog',
      'information_schema'
  )

GROUP BY
    t.table_schema,
    t.table_name

ORDER BY
    t.table_schema,
    t.table_name
"""


# ============================================================
# 5. MANIFEST TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (

    batch_id STRING,
    environment STRING,

    source_system STRING,
    source_database STRING,

    source_schema STRING,
    source_table STRING,

    bronze_table STRING,

    snapshot_type STRING,
    sample_percent INT,

    extracted_rows BIGINT,

    file_path STRING,

    extracted_at TIMESTAMP

)
USING DELTA

COMMENT 'Append-only PostgreSQL Bronze batch extraction manifest.'
""")


# ============================================================
# 6. DETERMINISTIC PRIMARY KEY SAMPLING
#
# Same primary key always selects the same DEV sample.
# No random sampling.
# ============================================================

def build_sample_predicate(primary_keys):

    key_parts = [
        sql.SQL("CAST({} AS TEXT)").format(
            sql.Identifier(column)
        )
        for column in primary_keys
    ]

    pk_expression = sql.SQL(
        "concat_ws('|', {})"
    ).format(
        sql.SQL(", ").join(key_parts)
    )

    predicate = sql.SQL("""
        mod(
            abs(
                (
                    'x' || substr(md5({}), 1, 8)
                )::bit(32)::bigint
            ),
            100
        ) < %s
    """).format(pk_expression)

    return predicate


# ============================================================
# 7. EXTRACT TABLES
# ============================================================

os.makedirs(
    VOLUME_ROOT,
    exist_ok=True
)

manifest_rows = []

table_results = []

print("=" * 65)
print("BANGLAPAY POSTGRESQL BRONZE EXTRACTION")
print("=" * 65)

print("Environment:", ENVIRONMENT.upper())
print("Batch ID:", BATCH_ID)
print("Data volume:", SAMPLE_PERCENT, "%")
print("Snapshot type:", SNAPSHOT_TYPE)


with psycopg.connect(
    PG_URL,

    user=PG_USERNAME,
    password=PG_PASSWORD,

    connect_timeout=15,

    sslmode="require",

    row_factory=dict_row

) as connection:

    # Keep all metadata and table reads consistent
    # within one read-only database snapshot.
    with connection.cursor() as cursor:

        cursor.execute("""
            SET TRANSACTION
            ISOLATION LEVEL REPEATABLE READ,
            READ ONLY
        """)

        cursor.execute(
            TABLE_INVENTORY_QUERY
        )

        tables = [
            dict(row)
            for row in cursor.fetchall()
        ]

    if len(tables) != EXPECTED_TABLE_COUNT:

        raise RuntimeError(
            f"Expected {EXPECTED_TABLE_COUNT} tables; "
            f"discovered {len(tables)}."
        )

    missing_pk = [
        f"{row['table_schema']}.{row['table_name']}"
        for row in tables
        if not row["primary_key_columns"]
    ]

    if missing_pk:
        raise RuntimeError(
            f"Tables without primary keys: {missing_pk}"
        )

    print("Discovered source tables:", len(tables))

    for index, table in enumerate(tables, start=1):

        schema_name = table["table_schema"]
        table_name = table["table_name"]

        primary_keys = table["primary_key_columns"]

        bronze_table = (
            f"pg_{schema_name}_{table_name}"
        )

        print(
            f"\n[{index}/{EXPECTED_TABLE_COUNT}] "
            f"{schema_name}.{table_name}"
        )

        # ----------------------------------------------------
        # Build deterministic selection predicate
        # ----------------------------------------------------

        predicate = build_sample_predicate(
            primary_keys
        )

        query = (
            sql.SQL("SELECT * FROM {}.{} WHERE ").format(
                sql.Identifier(schema_name),
                sql.Identifier(table_name)
            )
            + predicate
        )

        table_directory = (
            f"{VOLUME_ROOT}/"
            f"{schema_name}/"
            f"{table_name}/"
            f"batch_id={BATCH_ID}"
        )

        os.makedirs(
            table_directory,
            exist_ok=True
        )

        output_path = (
            f"{table_directory}/part-00000.ndjson"
        )

        extracted_count = 0

        try:

            with open(
                output_path,
                "x",
                encoding="utf-8"
            ) as output_file:

                cursor_name = (
                    f"bp_extract_{index}_"
                    f"{uuid.uuid4().hex[:8]}"
                )

                with connection.cursor(
                    name=cursor_name
                ) as cursor:

                    cursor.itersize = FETCH_SIZE

                    cursor.execute(
                        query,
                        (SAMPLE_PERCENT,)
                    )

                    while True:

                        rows = cursor.fetchmany(
                            FETCH_SIZE
                        )

                        if not rows:
                            break

                        for row in rows:

                            raw_record = dict(row)

                            source_pk = {
                                key: raw_record[key]
                                for key in primary_keys
                            }

                            envelope = {

                                "_meta": {

                                    "batch_id": BATCH_ID,

                                    "environment": ENVIRONMENT,

                                    "source_system": "neon_postgresql",

                                    "source_database": DATABASE_NAME,

                                    "source_schema": schema_name,

                                    "source_table": table_name,

                                    "source_primary_key": json.dumps(
                                        source_pk,
                                        default=json_safe
                                    ),

                                    "snapshot_type": SNAPSHOT_TYPE,

                                    "sample_percent": SAMPLE_PERCENT,

                                    "ingested_at": (
                                        EXTRACTED_AT.isoformat()
                                    )
                                },

                                "record": raw_record
                            }

                            output_file.write(
                                json.dumps(
                                    envelope,
                                    default=json_safe,
                                    ensure_ascii=False,
                                    separators=(",", ":")
                                )
                                + "\n"
                            )

                            extracted_count += 1

                output_file.flush()
                os.fsync(output_file.fileno())

        except Exception:

            # Never leave a partially extracted file
            # that a pipeline might accidentally ingest.
            try:
                os.remove(output_path)
            except FileNotFoundError:
                pass

            raise

        manifest_rows.append({

            "batch_id": BATCH_ID,

            "environment": ENVIRONMENT,

            "source_system": "neon_postgresql",

            "source_database": DATABASE_NAME,

            "source_schema": schema_name,

            "source_table": table_name,

            "bronze_table": bronze_table,

            "snapshot_type": SNAPSHOT_TYPE,

            "sample_percent": SAMPLE_PERCENT,

            "extracted_rows": extracted_count,

            "file_path": output_path,

            "extracted_at": EXTRACTED_AT
        })

        table_results.append({

            "source_schema": schema_name,

            "source_table": table_name,

            "bronze_table": bronze_table,

            "extracted_rows": extracted_count,

            "file_path": output_path
        })

        print(
            f"Extracted: {extracted_count:,} rows"
        )


# ============================================================
# 8. PERSIST MANIFEST
# ============================================================

MANIFEST_SCHEMA = StructType([

    StructField("batch_id", StringType(), False),

    StructField("environment", StringType(), False),

    StructField("source_system", StringType(), False),

    StructField("source_database", StringType(), False),

    StructField("source_schema", StringType(), False),

    StructField("source_table", StringType(), False),

    StructField("bronze_table", StringType(), False),

    StructField("snapshot_type", StringType(), False),

    StructField("sample_percent", IntegerType(), False),

    StructField("extracted_rows", LongType(), False),

    StructField("file_path", StringType(), False),

    StructField("extracted_at", TimestampType(), False)

])

manifest_data = [
    (
        row["batch_id"],
        row["environment"],
        row["source_system"],
        row["source_database"],
        row["source_schema"],
        row["source_table"],
        row["bronze_table"],
        row["snapshot_type"],
        row["sample_percent"],
        row["extracted_rows"],
        row["file_path"],
        row["extracted_at"]
    )
    for row in manifest_rows
]

manifest_df = spark.createDataFrame(
    manifest_data,
    schema=MANIFEST_SCHEMA
)

(
    manifest_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(MANIFEST_TABLE)
)


# ============================================================
# 9. FINAL REPORT
# ============================================================

total_extracted = sum(
    row["extracted_rows"]
    for row in table_results
)

print("\n" + "=" * 65)
print("POSTGRESQL INITIAL SNAPSHOT COMPLETED")
print("=" * 65)

print("Batch ID:", BATCH_ID)
print("Tables extracted:", len(table_results))
print("Total extracted rows:", f"{total_extracted:,}")

print("Volume:", SAMPLE_PERCENT, "%")

print("Manifest:", MANIFEST_TABLE)

display(
    spark.createDataFrame(table_results)
    .orderBy("source_schema", "source_table")
)

print("SOURCE DATA WAS NOT MODIFIED.")